## Lab Assignment - 3

#### Fine-tune GPT or GPT-2 for creative story generation.


In [ ]:
# install required libraries
!pip install transformers datasets accelerate -q

In [ ]:
# import libraries
import pandas as pd
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

In [ ]:
# creating a small story dataset
stories = [
    "Once upon a time in a magical forest, a small rabbit discovered a glowing stone that could talk.",
    "In a distant galaxy, a lonely robot learned the meaning of friendship when a child visited his planet.",
    "A young girl found a secret door in her grandmother's house that led to a world of dragons.",
    "On a rainy evening, a detective received a mysterious letter that changed his life forever.",
    "Deep under the ocean, a curious dolphin discovered an ancient underwater city."
]

df = pd.DataFrame(stories, columns=["text"])

dataset = Dataset.from_pandas(df)
dataset

In [ ]:
# Load GPT-2 Model and Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 has no pad token, so set it
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
# Tokenize the Dataset
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
# Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
# Training Configuration
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-story",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2
)

In [ ]:
# tarier step
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
# tarin the model
trainer.train()

In [ ]:
# generate a story
prompt = "Once upon a time"

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs["input_ids"],
    max_length=100,
    num_return_sequences=1,
    temperature=0.9,
    top_k=50
)

generated_story = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_story)

In [ ]:
# saving the model
model.save_pretrained("story_generator_model")
tokenizer.save_pretrained("story_generator_model")